# Data exploration — the real credit datasets

This notebook looks at **what we are generating a prior *for***. It is the companion to
`prior_visualisation.ipynb`, which looks at what the prior produces.

Read together, the two answer the question the project rests on:

> Does our synthetic prior look like credit data in the ways that **matter** — and not
> only in the ways that are easy to hit?

Three measurements drive everything:

| what | why it matters |
|---|---|
| **boundary mass** (LGD) | the share of targets sitting *exactly* at 0 or 1. The original TabICL prior produces essentially none. This is the gap the project exists to fill. |
| **base rate** (PD) | how rare default is. TabICL's prior is roughly balanced; real data is not. |
| **shape and type mix** | rows, columns, categorical share, missingness. These set the ranges the prior samples over. |

All numbers come from the **processed parquet cache** through `src.data.pipeline`, so this
notebook sees exactly the tables the evaluation sees. Any dataset not yet processed is
processed on first access, which can take a few minutes the first time.

Every plot lives in `src/visualize/data_plots.py`; this notebook holds no logic.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.visualize import data_plots, style

style.use_style()          # one visual language for every figure in the project
pd.set_option("display.width", 200, "display.max_columns", 40)
style.show_palette();

## 1. Load everything

`load_all` preprocesses anything missing from the cache, then loads it. A dataset that
cannot be read is **skipped with a message** rather than killing the notebook — one bad
raw file should not stop you looking at the other twenty.

In [ ]:
lgd = data_plots.load_all("lgd")
pd_ = data_plots.load_all("pd")
print(f"\nloaded {len(lgd)} LGD and {len(pd_)} PD datasets")

## 2. LGD at a glance

The columns to read first are **`boundary mass`** and **`in [0,1]`**.

`boundary mass` is the fraction of rows sitting exactly at the minimum or maximum. Where
it is large, a model that can only produce smooth interior predictions is structurally
unable to be calibrated — it can get the *average* LGD right while being wrong about every
individual loan.

In [ ]:
lgd_summary = data_plots.summary_table(lgd, "lgd")
lgd_summary

### 2a. Every LGD target, one panel each

The motivating figure of the whole project. Compare the two end bars against the middle.

Then look at the equivalent plot for the **original** prior in `prior_visualisation.ipynb`
section 7: it has no end bars at all.

In [ ]:
fig = data_plots.plot_lgd_targets(lgd)

### 2b. Which datasets does boundary mass actually matter for?

A ranked list, because that is the practical question. Two things to take from it:

* the spread is **wide** — so the prior has to produce a *range* of boundary masses, not
  one representative value. That is why `atom_prob` is a swept lever and not a constant.
* mass at 0 (full recovery) and mass at 1 (total loss) are **not symmetric**, so the prior
  samples them separately.

In [ ]:
fig = data_plots.plot_boundary_mass_ranking(lgd)

## 3. PD at a glance

For PD the interesting quantity is the **base rate**: how rare default is. `imbalance 1:n`
is the same number stated as odds, which is easier to feel — 1:99 means one default per
hundred loans.

In [ ]:
pd_summary = data_plots.summary_table(pd_, "pd")
pd_summary

In [ ]:
fig = data_plots.plot_pd_base_rates(pd_)

The dashed grey line is roughly where TabICL's prior sits. **Every real dataset is to the
left of it**, most by a long way.

This is why `base_rate_range` exists in `config/PD.yaml` and why PD gets its own target
mechanism: a prior trained near balance has never had to learn that the interesting class
can be 1% of the data.

## 4. Shape and type — where the prior's ranges come from

If the prior generated 500-column tables and every real dataset had 20, the extra capacity
would be wasted; if it generated 200-row tables and real ones have a million, it would
never learn to use a long context. So these plots set `n_rows_range`, `n_features_range`
and the categorical fraction.

In [ ]:
both = {"lgd": lgd, "pd": pd_}
fig = data_plots.plot_shapes(both)

In [ ]:
fig = data_plots.plot_type_mix(both)

In [ ]:
fig = data_plots.plot_missingness(both)

Note how many datasets have **zero** missing values. Most were imputed before we received
them, so real missingness is understated here. Our prior still injects missingness as an
explicit mechanism, because a model that has never seen a NaN handles one badly — but
these datasets are not the evidence for how much to inject.

## 5. Feature dependence — the part O'Prior says actually transfers

O'Prior's argument is that what a prior teaches is a **dependence structure**, not
individual functions. So it matters what real credit data looks like: blocks of strongly
correlated columns (several measures of the same balance, several vintages of the same
delinquency count), not independent features.

Our prior builds features through random DAGs precisely so that correlated blocks appear.
If these heatmaps were diagonal, that design choice would be wrong.

In [ ]:
fig = data_plots.plot_feature_correlations(lgd, n_show=6)

In [ ]:
fig = data_plots.plot_feature_correlations(pd_, n_show=6)

## 6. Leakage screen

A cheap single-feature check: for every column, its absolute correlation with the target.

This exists because of a real finding — **`lgd_lendingclub` gives R² around 0.71–0.76**,
far above anything reported for LGD modelling, which usually means a column encodes the
answer. Anything flagged `suspicious` deserves a look at the raw file before that dataset
is used to support a claim.

A high correlation is a **pointer, not a proof**. A single strong predictor can be
legitimate.

In [ ]:
flags = data_plots.leakage_check(lgd, "lgd")
flags.head(15)

In [ ]:
flags[flags["suspicious"]]

## 7. What this means for the prior

Pulling the measurements together:

1. **LGD boundary mass spans a wide range across datasets.** So the prior samples
   `atom_prob` over a family rather than fixing one value — matching a single dataset's
   boundary mass would be overfitting to that dataset.
2. **LGD targets are genuinely bounded.** Clipping to [0,1] is not a cosmetic step; it
   encodes a real constraint the original prior does not have.
3. **PD base rates sit well below balance, across two orders of magnitude.** So imbalance
   is sampled, not fixed.
4. **Real features come in correlated blocks.** DAG-based generation is the right shape;
   independent features would not reproduce this.
5. **Missingness is mostly pre-imputed away.** Keep injecting it, but do not tune its rate
   to these numbers — they measure the upstream pipeline, not the domain.

Points 1–3 are what `config/LGD.yaml` and `config/PD.yaml` sweep. See
`docs/experimental_design.md` for how the arms are laid out.